#Import Dataset from Drive



In [4]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os

folder = "/content/drive/MyDrive/Studio2_DeepLearning"

print("Folder ditemukan:", os.path.exists(folder))
print("\nIsi folder Google Drive:")

if os.path.exists(folder):
    for file in os.listdir(folder):
        print(file)

Folder ditemukan: True

Isi folder Google Drive:
starter_kit-20261011T124642Z-1-001.zip
studio2-sampah.zip


#Ekstrak Training Baseline (ResNet-18)

In [6]:
import zipfile
from pathlib import Path

drive_folder = Path("/content/drive/MyDrive/Studio2_DeepLearning")
project = Path("/content/studio2-sampah")

project.mkdir(parents=True, exist_ok=True)

# 1. Ekstrak dataset
dataset_zip = drive_folder / "studio2-sampah.zip"

with zipfile.ZipFile(dataset_zip, "r") as z:
    z.extractall("/content/")

print("Dataset berhasil diekstrak!")

# 2. Ekstrak starter kit
starter_zip = drive_folder / "starter_kit-20261011T124642Z-1-001.zip"

with zipfile.ZipFile(starter_zip, "r") as z:
    z.extractall(project)

print("Starter kit berhasil diekstrak!")

# 3. Periksa hasil
print("\nIsi folder proyek:")
for item in project.iterdir():
    print("-", item.name)

print("\nLokasi kode training:")
for file in project.rglob("train_baseline.py"):
    print(file)

Dataset berhasil diekstrak!
Starter kit berhasil diekstrak!

Isi folder proyek:
- starter_kit
- train

Lokasi kode training:
/content/studio2-sampah/starter_kit/train_baseline.py


In [7]:
from pathlib import Path

train_dir = Path("/content/studio2-sampah/train")

total = 0

for folder in sorted(train_dir.iterdir()):
    if folder.is_dir():
        jumlah = len(list(folder.glob("*.jpg")))
        print(f"{folder.name}: {jumlah} gambar")
        total += jumlah

print(f"\nTotal dataset: {total} gambar")

botol_plastik: 523 gambar
gelas_plastik: 521 gambar
kertas_kardus: 365 gambar
saset_kemasan: 673 gambar
styrofoam: 671 gambar

Total dataset: 2753 gambar


In [ ]:
#

In [11]:
%cd /content/studio2-sampah/starter_kit

/content/studio2-sampah/starter_kit


#Eksekusi train_baseline.py train dengan 8 epoch

In [12]:
!python -u train_baseline.py --data ../train --epochs 8

kelas: ['botol_plastik', 'gelas_plastik', 'kertas_kardus', 'saset_kemasan', 'styrofoam'] | train: 2343 | val: 410
epoch  1  val_acc=0.8098
epoch  2  val_acc=0.8976
epoch  3  val_acc=0.9195
epoch  4  val_acc=0.9220
epoch  5  val_acc=0.9171
epoch  6  val_acc=0.9341
epoch  7  val_acc=0.9268
epoch  8  val_acc=0.9293
best val_acc=0.9341 -> submission/model.pth


#Eksekusi code Evaluate.py untuk model ResNet-18

In [13]:
!python evaluate.py --submission ./submission --data ./my_val

device=cuda  gambar=410

Macro-F1 : 0.9294
Akurasi  : 0.9341
  F1 botol_plastik   0.9474
  F1 gelas_plastik   0.9554
  F1 kertas_kardus   0.8713
  F1 saset_kemasan   0.9307
  F1 styrofoam       0.9423
Confusion matrix (baris = label asli, kolom = prediksi):
  botol_plastik   [72, 1, 0, 2, 3]
  gelas_plastik   [1, 75, 0, 1, 1]
  kertas_kardus   [0, 2, 44, 4, 4]
  saset_kemasan   [1, 1, 2, 94, 2]
  styrofoam       [0, 0, 1, 1, 98]


#Penyimpanan Model Baseline ke drive

In [14]:
import shutil
from pathlib import Path

source = Path(
    "/content/studio2-sampah/starter_kit/submission"
)

backup = Path(
    "/content/drive/MyDrive/Studio2_DeepLearning/baseline_resnet18"
)

backup.mkdir(parents=True, exist_ok=True)

shutil.copy2(source / "model.pth", backup / "model.pth")
shutil.copy2(source / "model.py", backup / "model.py")

print("Baseline ResNet-18 berhasil disimpan!")

Baseline ResNet-18 berhasil disimpan!


#Inisialisasi EfficientNet-B2 dari Hugging Face

In [19]:
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification
)

model_name = "google/efficientnet-b2"

processor = AutoImageProcessor.from_pretrained(model_name)

model = AutoModelForImageClassification.from_pretrained(
    model_name,
    num_labels=5,
    ignore_mismatched_sizes=True
)

print("EfficientNet-B2 berhasil dimuat!")
print("Jumlah kelas:", model.config.num_labels)

preprocessor_config.json:   0%|          | 0.00/495 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/70.2k [00:00<?, ?B/s]

[transformers] You passed `num_labels=5` which is incompatible to the `id2label` map of length `1000`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 36.9MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/508 [00:00<?, ?it/s]

[transformers] EfficientNetForImageClassification LOAD REPORT from: google/efficientnet-b2
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 1408]) vs model:torch.Size([5, 1408])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([5])            

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


EfficientNet-B2 berhasil dimuat!
Jumlah kelas: 5


#Pemeriksaan GPU dan Output Model

In [20]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)
model.eval()

x = torch.randn(2, 3, 260, 260).to(device)

with torch.no_grad():
    output = model(pixel_values=x)

print("GPU:", device)
print("Output:", output.logits.shape)

GPU: cuda
Output: torch.Size([2, 5])


#Persiapan Data dan Augmentasi

In [34]:
from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

base = Path("/content/studio2-sampah/starter_kit")

train_dir = base / "my_train"
val_dir = base / "my_val"

# Transformasi training: augmentasi gambar
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(260, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=processor.image_mean,
        std=processor.image_std
    )
])

# Transformasi validasi: tanpa augmentasi acak
val_transform = transforms.Compose([
    transforms.Resize(288),
    transforms.CenterCrop(260),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=processor.image_mean,
        std=processor.image_std
    )
])

train_dataset = datasets.ImageFolder(
    train_dir, transform=train_transform
)

val_dataset = datasets.ImageFolder(
    val_dir, transform=val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2
)

print("Kelas:", train_dataset.classes)
print("Jumlah training:", len(train_dataset))
print("Jumlah validasi:", len(val_dataset))
print("Jumlah batch training:", len(train_loader))

Kelas: ['botol_plastik', 'gelas_plastik', 'kertas_kardus', 'saset_kemasan', 'styrofoam']
Jumlah training: 2343
Jumlah validasi: 410
Jumlah batch training: 147


#Reset EfficientNet-B2 ke Bobot Pretrained

In [35]:
from transformers import AutoModelForImageClassification

model = AutoModelForImageClassification.from_pretrained(
    "google/efficientnet-b2",
    num_labels=5,
    ignore_mismatched_sizes=True
)

model = model.to(device)

print("EfficientNet-B2 berhasil diinisialisasi ulang!")

[transformers] You passed `num_labels=5` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/508 [00:00<?, ?it/s]

[transformers] EfficientNetForImageClassification LOAD REPORT from: google/efficientnet-b2
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 1408]) vs model:torch.Size([5, 1408])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([5])            

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


EfficientNet-B2 berhasil diinisialisasi ulang!


# Fine-Tuning EfficientNet-B2




In [36]:
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, accuracy_score
from pathlib import Path

# Pengaturan training
epochs = 8
learning_rate = 0.0001

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Folder khusus EfficientNet-B2
save_dir = Path(
    "/content/drive/MyDrive/Studio2_DeepLearning/efficientnet_b2"
)
save_dir.mkdir(parents=True, exist_ok=True)

best_f1 = -1.0

for epoch in range(epochs):

    # TRAINING
    model.train()
    total_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(pixel_values=images)
        loss = criterion(outputs.logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    scheduler.step()

    # VALIDASI
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)

            outputs = model(pixel_values=images)
            predictions = outputs.logits.argmax(dim=1)

            all_preds.extend(predictions.cpu().tolist())
            all_labels.extend(labels.tolist())

    val_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    val_acc = accuracy_score(all_labels, all_preds)
    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1:02d}/{epochs} | "
        f"Loss: {avg_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro-F1: {val_f1:.4f}",
        flush=True
    )

    if val_f1 > best_f1:
        best_f1 = val_f1

        torch.save(
            model.state_dict(),
            save_dir / "model_best.pth"
        )

        print("  -> Model terbaik disimpan!", flush=True)

print("\nTraining selesai!")
print(f"Best Macro-F1: {best_f1:.4f}")
print("Lokasi model:", save_dir / "model_best.pth")

Epoch 01/8 | Loss: 1.0193 | Val Acc: 0.8902 | Macro-F1: 0.8865
  -> Model terbaik disimpan!
Epoch 02/8 | Loss: 0.6257 | Val Acc: 0.9146 | Macro-F1: 0.9106
  -> Model terbaik disimpan!
Epoch 03/8 | Loss: 0.5290 | Val Acc: 0.9293 | Macro-F1: 0.9262
  -> Model terbaik disimpan!
Epoch 04/8 | Loss: 0.5069 | Val Acc: 0.9439 | Macro-F1: 0.9412
  -> Model terbaik disimpan!
Epoch 05/8 | Loss: 0.4699 | Val Acc: 0.9341 | Macro-F1: 0.9289
Epoch 06/8 | Loss: 0.4624 | Val Acc: 0.9366 | Macro-F1: 0.9336
Epoch 07/8 | Loss: 0.4617 | Val Acc: 0.9390 | Macro-F1: 0.9355
Epoch 08/8 | Loss: 0.4541 | Val Acc: 0.9049 | Macro-F1: 0.9016

Training selesai!
Best Macro-F1: 0.9412
Lokasi model: /content/drive/MyDrive/Studio2_DeepLearning/efficientnet_b2/model_best.pth


#Pembuatan File Submission EfficientNet-B2

In [38]:
from pathlib import Path
import shutil

from transformers import (
    EfficientNetConfig,
    EfficientNetForImageClassification
)

submission_b2 = Path(
    "/content/studio2-sampah/starter_kit/submission_b2"
)
submission_b2.mkdir(parents=True, exist_ok=True)

# Simpan konfigurasi arsitektur dari model yang dilatih
model.config.save_pretrained(submission_b2)

model_code = '''
from pathlib import Path
from torchvision import transforms
from transformers import (
    EfficientNetConfig,
    EfficientNetForImageClassification
)

class SampahEfficientNetB2(EfficientNetForImageClassification):
    def forward(self, x):
        return super().forward(pixel_values=x).logits

def get_model():
    config_path = Path(__file__).resolve().parent / "config.json"
    config = EfficientNetConfig.from_json_file(str(config_path))
    return SampahEfficientNetB2(config)

def get_transform():
    return transforms.Compose([
        transforms.Resize(288),
        transforms.CenterCrop(260),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.47853944, 0.4732864, 0.47434163]
        )
    ])
'''

(submission_b2 / "model.py").write_text(model_code)

source = Path(
    "/content/drive/MyDrive/Studio2_DeepLearning/efficientnet_b2/model_best.pth"
)

shutil.copy2(source, submission_b2 / "model.pth")

print("File submission:")
for file in submission_b2.iterdir():
    print("-", file.name)

File submission:
- model.py
- config.json
- model.pth


#EKSEKUSI CODE EVALUATE UNTUK EFFICIENT NET

In [40]:
%cd /content/studio2-sampah/starter_kit

!python evaluate.py --submission ./submission_b2 --data ./my_val

/content/studio2-sampah/starter_kit
device=cuda  gambar=410

Macro-F1 : 0.9412
Akurasi  : 0.9439
  F1 botol_plastik   0.9806
  F1 gelas_plastik   0.9494
  F1 kertas_kardus   0.8952
  F1 saset_kemasan   0.9442
  F1 styrofoam       0.9366
Confusion matrix (baris = label asli, kolom = prediksi):
  botol_plastik   [76, 1, 0, 0, 1]
  gelas_plastik   [0, 75, 1, 0, 2]
  kertas_kardus   [0, 2, 47, 3, 2]
  saset_kemasan   [1, 0, 2, 93, 4]
  styrofoam       [0, 2, 1, 1, 96]


#Backup Submission ke Google Drive

In [41]:
import shutil
from pathlib import Path

source = Path(
    "/content/studio2-sampah/starter_kit/submission_b2"
)

destination = Path(
    "/content/drive/MyDrive/Studio2_DeepLearning/submission_b2"
)

destination.mkdir(parents=True, exist_ok=True)

for filename in ["model.py", "config.json", "model.pth"]:
    shutil.copy2(source / filename, destination / filename)

print("Backup submission berhasil!")
for file in destination.iterdir():
    print(file.name)

Backup submission berhasil!
model.py
config.json
model.pth
